# Linear regression on the different housing datasets (California and Ames)


In [ ]:
# a lot of imports
%matplotlib inline
import sklearn
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split


print(sklearn.__version__)

**Recap: The California Housing Dataset**

In [ ]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()

X,y = housing.data, housing.target
X.shape

In [ ]:
housing.feature_names

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)
print("Size of training data: ", X_train.shape)
print("Size of test data: ", X_test.shape)


**Apply Linear regression**

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
lr

In [ ]:
# look at model parameters! Attributes: coef_ and intercept_
print("Coefficients of the model:", lr.coef_)
print("Y-Intercept of the model:",lr.intercept_)


Letz calculate the hypothesis function like given on the lecture slides
$h_{\theta}(x) = \theta^T \cdot x \quad \text{with} \quad 
x = \begin{pmatrix}
1 \\
x^{(1)} \\
\vdots \\
x^{(j)}
\end{pmatrix}
\quad \text{and} \quad 
\theta = \begin{pmatrix}
\theta_0 \\
\theta_1 \\
\vdots \\
\theta_j
\end{pmatrix}$

In [ ]:
#Make a prediction - first manually 
value = X_test[0].dot(lr.coef_)+lr.intercept_
print(f"The value is {value*100000:.2f} $")

In [ ]:
#Make a prediction - now directly with the model  
pickedSamples = X_test[0:1]
print("PickedSamples.shape",pickedSamples.shape) 

In [ ]:
print("The prediced value was");
print(lr.predict(pickedSamples))
print("The real value was");
print(y_test[0:1])


In [ ]:
print("Training set score: {:.2f}".format(lr.score(X_train, y_train)))
print("Test set score: {:.2f}".format(lr.score(X_test, y_test)))

Question: Interpret these values and compare with results from previous lectures? How can the result be explained?
<details>
    <summary><i>Solution</i></summary>
The model is underfitting. In KNN we used to have better results. This is because nolinear correlations between variables and output cannot be learned by the model. 
</details>

## A different dataset:  Ames Housing Extended

This is a housing dataset which provides much more features but way less examples. So our model is more likely to overfit.

In [ ]:
# --- 1. Imports ---
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score

# --- 2. Load Ames dataset (real-world, not synthetic) ---
X, y = fetch_openml(name="house_prices", as_frame=True, return_X_y=True)

In [ ]:
# Inspect the data
X

In [ ]:
#y = y.astype(float) # convert labels to float to have real continuous values for prediction
y

## Data Preprocessing
Looking at the data we saw we have some data that cannot be treated by machine learning algorithms well. There are categorical values (not numbers) and also some numeric values have no value given (NaN = not a number).

In [ ]:
# --- 3. Basic preprocessing ---
# Separate numeric and categorical columns
num_cols = X.select_dtypes(include=["number"]).columns
cat_cols = X.select_dtypes(exclude=["number"]).columns
cat_cols

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

# In the following we normalize numerical data and one-hot encode categorical input values
featureMatrix = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
   ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
]).fit_transform(X)
featureMatrix.shape

# second preprocessing replacing NaN
featureMatrix = np.nan_to_num(featureMatrix, nan=0.0)
featureMatrix.shape

In [ ]:
# --- 4. Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(
    featureMatrix, y, random_state=100
)

# --- 5. Models ---
lr = LinearRegression()
lr.fit(X_train, y_train)

print("Train R²:", r2_score(y_train, lr.predict(X_train)))
print("Test  R²:", r2_score(y_test,  lr.predict(X_test)))

Question: Are we observing over- or underfitting ? What could we do to improve results?
<details>
    <summary><i>Solution</i></summary>
The model is overfitting. There are many parameters so many parameter configurations are possible. 
If we have many parameters each parameter can be dedicated to few input variables and be tuned in a way that it the result is 
excessively sensitive to small changes in input variables. This effect that is caused by high cooeficient values to emphasize certain input variables and also causes 
the model to overfit.
This for a less complex model we should create an incentive to keep the model weights small!
</details>

**Ridge Regression on Ames Housing - Introduction of L2-"Regularization"**

In Ridge Regression we have an additional goal. We want to be the coefficients to be as small as possible. We do this by adding a penalty for having large coefficients.

$\Large{J(\theta_0, \theta_1) = \frac{1}{2m}\sum_{i=1}^{m}{(h_\theta(x[i])-y[i])^2} + \alpha \lVert \theta \rVert_2^2}$


$\alpha$ helps to determine how well a model should be regularized. Greater values means more regularization (less complex model).


In [ ]:
# Alpha=1
from sklearn.linear_model import Ridge

ridge = Ridge().fit(X_train, y_train)
print("Training set score: {:.2f}".format(ridge.score(X_train, y_train)))
print("Test set score: {:.2f}".format(ridge.score(X_test, y_test)))

In [ ]:
# inspect the alpha value
ridge.alpha

In [ ]:
ridge100 = Ridge(alpha=100).fit(X_train, y_train) # stronger regularization - additional cost for higher cooeficients
print("Training set score: {:.2f}".format(ridge100.score(X_train, y_train)))
print("Test set score: {:.2f}".format(ridge100.score(X_test, y_test)))

In [ ]:
ridge01 = Ridge(alpha=0.1).fit(X_train, y_train)
print("Training set score: {:.2f}".format(ridge01.score(X_train, y_train)))
print("Test set score: {:.2f}".format(ridge01.score(X_test, y_test)))

In [ ]:
ridge20000 = Ridge(alpha=20000).fit(X_train, y_train)
print("Training set score: {:.2f}".format(ridge20000.score(X_train, y_train)))
print("Test set score: {:.2f}".format(ridge20000.score(X_test, y_test)))

<b>Plot the coefficients of all trained regressors</b>

In [ ]:
plt.figure(figsize=(12, 12))

# we just look at the first 100 coefficients
plt.plot(lr.coef_[0:100], 'o', label="LinearRegression    Test Score:{:.2f}".format(lr.score(X_test, y_test)))
plt.plot(ridge01.coef_[0:100], 'v', label="Ridge alpha=0.1    Test Score:{:.2f}".format(ridge01.score(X_test, y_test)))
plt.plot(ridge.coef_[0:100], 's', label="Ridge alpha=1      Test Score:{:.2f}".format(ridge.score(X_test, y_test)))
plt.plot(ridge100.coef_[0:100], '^', label="Ridge alpha=100    Test Score:{:.2f}".format(ridge10.score(X_test, y_test)))

plt.xlabel("Coefficient index")
plt.ylabel("Coefficient magnitude")
xlims = plt.xlim()
plt.hlines(0, xlims[0], xlims[1])
plt.xlim(xlims)
plt.ylim(-10000, 45000)
plt.legend()
plt.show()